# 06 — Targeted post-freeze strengthening analyses

Public source-only reproducibility notebook. Outputs and execution history were removed. No credentials, patient-level data, row-level predictions, or row-level SHAP values are included. Execution requires credentialed access to the eICU Collaborative Research Database and an authorized Google Cloud project.


In [ ]:
import os
from pathlib import Path

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
WORK_DATASET_NAME = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
SOURCE_DATASET = os.environ.get("EICU_SOURCE_DATASET", "physionet-data.eicu_crd")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")
OUTPUT_ROOT = os.environ.get("AKI_OUTPUT_ROOT", "/content/AKI_JCMC_V2_PUBLIC_RUN")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

print("Target dataset:", TARGET_DATASET)
print("Output root:", OUTPUT_ROOT)


# AKI V2 — Targeted Strengthening Runner v001

This notebook adds only the four high-yield aggregate analyses that are not already frozen:

1. alternative reference-creatinine label sensitivity;
2. landmark inclusion/exclusion characterization;
3. prespecified operating-point and alarm-burden table;
4. hospital-level performance and missingness heterogeneity.

## Safety rules

- No primary model retuning or reselection.
- No patient-level data are written to Google Drive.
- Existing locked XGBoost predictions are read from BigQuery.
- All saved outputs are aggregate tables or figures.
- The primary validation remains **hospital-disjoint internal–external cross-validation**.

In [ ]:
# 00 — Authentication and fixed configuration
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import os, json, hashlib, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.cloud import bigquery
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
DATASET_ID = "aki_jcmc_v2"
TARGET_DATASET = f"{PROJECT_ID}.{DATASET_ID}"
BQ_LOCATION = "US"

client = bigquery.Client(project=PROJECT_ID)

ROOT = Path(f"{OUTPUT_ROOT}")
OUT = ROOT / "05_MODELING_OUTPUTS" / "41_STRENGTHENING"
OUT.mkdir(parents=True, exist_ok=True)

PRED_TABLE = f"{TARGET_DATASET}.model_xgb_outer_predictions_all5_v1"
CORE_TABLE = f"{TARGET_DATASET}.feature_matrix_core_outerfold_v1"
COHORT_TABLE = f"{TARGET_DATASET}.cohort_outcome_v1"
CREAT_TABLE = f"{TARGET_DATASET}.creatinine_staged_v1"
BASE_TABLE = f"{TARGET_DATASET}.base_stays_v1"
PATIENT_TABLE = "physionet-data.eicu_crd.patient"

LANDMARK_MIN = 720
OUTCOME_END_MIN = 4320
EXPECTED_N = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198

def save_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False)
    print("saved:", path)
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("Target dataset:", TARGET_DATASET)
print("Output directory:", OUT)

## 01 — Locked-source integrity checks

This cell verifies the frozen prediction table and the hospital-disjoint core matrix before any strengthening result is calculated.

In [ ]:
sql_integrity = f'''
SELECT
  COUNT(*) AS row_count,
  COUNTIF(label_stage23 = 1) AS events,
  COUNT(DISTINCT id_row) AS distinct_rows
FROM `{PRED_TABLE}`
'''
integrity = client.query(sql_integrity, location=BQ_LOCATION).to_dataframe()

sql_core = f'''
SELECT
  COUNT(*) AS row_count,
  COUNTIF(label_stage23 = 1) AS events,
  COUNT(DISTINCT group_hospital) AS hospitals,
  COUNT(DISTINCT id_row) AS distinct_rows
FROM `{CORE_TABLE}`
'''
core_integrity = client.query(sql_core, location=BQ_LOCATION).to_dataframe()

display(integrity)
display(core_integrity)

assert int(integrity.loc[0, "row_count"]) == EXPECTED_N
assert int(integrity.loc[0, "events"]) == EXPECTED_EVENTS
assert int(integrity.loc[0, "distinct_rows"]) == EXPECTED_N

assert int(core_integrity.loc[0, "row_count"]) == EXPECTED_N
assert int(core_integrity.loc[0, "events"]) == EXPECTED_EVENTS
assert int(core_integrity.loc[0, "hospitals"]) == EXPECTED_HOSPITALS
assert int(core_integrity.loc[0, "distinct_rows"]) == EXPECTED_N

print("Locked-source integrity: PASS")

## 02 — Reference-creatinine label-sensitivity analysis

The primary cohort is kept fixed. Alternative reference rules are used only to quantify phenotype sensitivity.

Alternative rules:

- `locked_current`: current earliest value between hospital admission/−24 h and +6 h ICU;
- `pre_icu_earliest`: earliest qualifying value no later than ICU admission;
- `first_icu_0_360`: first value from ICU admission through +6 h;
- `minimum_prelandmark`: minimum value available through the 12 h landmark.

The last rule is a sensitivity phenotype only and must not be described as a universally correct baseline.

In [ ]:
sql_reference_sensitivity = f'''
WITH main AS (
  SELECT
    c.patientUnitStayID,
    c.outcome_creatinine_stage23 AS locked_label,
    c.reference_creatinine AS locked_reference,
    c.max_stage_by_12h AS locked_max_stage_by_12h,
    c.unitDischargeOffset,
    b.hospitalAdmitOffset
  FROM `{COHORT_TABLE}` c
  JOIN `{BASE_TABLE}` b USING(patientUnitStayID)
  WHERE c.eligible_main_cohort = 1
    AND c.deterministic_eligible_patient_stay_rank = 1
),
refs AS (
  SELECT
    m.patientUnitStayID,
    ANY_VALUE(m.locked_label) AS locked_label,
    ANY_VALUE(m.locked_reference) AS locked_reference,
    ARRAY_AGG(
      IF(
        s.labResultOffset BETWEEN GREATEST(m.hospitalAdmitOffset, -1440) AND 0,
        STRUCT(s.labResultOffset AS offset, s.creatinine AS value),
        NULL
      )
      IGNORE NULLS ORDER BY s.labResultOffset LIMIT 1
    )[SAFE_OFFSET(0)] AS pre_icu_earliest,
    ARRAY_AGG(
      IF(
        s.labResultOffset BETWEEN 0 AND 360,
        STRUCT(s.labResultOffset AS offset, s.creatinine AS value),
        NULL
      )
      IGNORE NULLS ORDER BY s.labResultOffset LIMIT 1
    )[SAFE_OFFSET(0)] AS first_icu_0_360,
    MIN(IF(s.labResultOffset BETWEEN GREATEST(m.hospitalAdmitOffset, -1440) AND 720,
           s.creatinine, NULL)) AS minimum_prelandmark
  FROM main m
  LEFT JOIN `{CREAT_TABLE}` s USING(patientUnitStayID)
  GROUP BY m.patientUnitStayID
),
long_refs AS (
  SELECT patientUnitStayID, locked_label, 'locked_current' AS method,
         locked_reference AS reference_creatinine
  FROM refs
  UNION ALL
  SELECT patientUnitStayID, locked_label, 'pre_icu_earliest',
         pre_icu_earliest.value
  FROM refs
  UNION ALL
  SELECT patientUnitStayID, locked_label, 'first_icu_0_360',
         first_icu_0_360.value
  FROM refs
  UNION ALL
  SELECT patientUnitStayID, locked_label, 'minimum_prelandmark',
         minimum_prelandmark
  FROM refs
),
staged AS (
  SELECT
    r.patientUnitStayID,
    r.method,
    r.locked_label,
    r.reference_creatinine,
    s.labResultOffset,
    s.creatinine,
    s.prior_min_creatinine_48h,
    CASE
      WHEN r.reference_creatinine IS NULL THEN NULL
      WHEN SAFE_DIVIDE(s.creatinine, r.reference_creatinine) >= 3.0
        OR (
          s.creatinine >= 4.0
          AND (
            SAFE_DIVIDE(s.creatinine, r.reference_creatinine) >= 1.5
            OR (
              s.prior_min_creatinine_48h IS NOT NULL
              AND s.creatinine - s.prior_min_creatinine_48h >= 0.3
            )
          )
        ) THEN 3
      WHEN SAFE_DIVIDE(s.creatinine, r.reference_creatinine) >= 2.0 THEN 2
      WHEN SAFE_DIVIDE(s.creatinine, r.reference_creatinine) >= 1.5
        OR (
          s.prior_min_creatinine_48h IS NOT NULL
          AND s.creatinine - s.prior_min_creatinine_48h >= 0.3
        ) THEN 1
      ELSE 0
    END AS alt_stage
  FROM long_refs r
  LEFT JOIN `{CREAT_TABLE}` s USING(patientUnitStayID)
),
patient_method AS (
  SELECT
    patientUnitStayID,
    method,
    ANY_VALUE(locked_label) AS locked_label,
    ANY_VALUE(reference_creatinine) AS reference_creatinine,
    MAX(IF(labResultOffset <= 720, alt_stage, NULL)) AS max_stage_by_12h,
    MAX(IF(labResultOffset > 720 AND labResultOffset <= 4320, alt_stage, NULL))
      AS max_stage_12_72h
  FROM staged
  GROUP BY patientUnitStayID, method
),
classified AS (
  SELECT
    *,
    CASE
      WHEN reference_creatinine IS NULL THEN NULL
      WHEN COALESCE(max_stage_by_12h, 0) >= 2 THEN NULL
      WHEN max_stage_12_72h >= 2 THEN 1
      ELSE 0
    END AS alternative_label
  FROM patient_method
)
SELECT
  method,
  COUNT(*) AS primary_cohort_patients,
  COUNTIF(reference_creatinine IS NOT NULL) AS reference_available,
  COUNTIF(COALESCE(max_stage_by_12h, 0) >= 2) AS severe_by_landmark,
  COUNTIF(alternative_label IS NOT NULL) AS alternative_eligible,
  COUNTIF(alternative_label = 1) AS alternative_events,
  SAFE_DIVIDE(COUNTIF(alternative_label = 1),
              COUNTIF(alternative_label IS NOT NULL)) AS alternative_event_rate,
  COUNTIF(alternative_label IS NOT NULL AND alternative_label = locked_label)
    AS labels_agree,
  COUNTIF(alternative_label IS NOT NULL AND alternative_label != locked_label)
    AS labels_disagree,
  COUNTIF(alternative_label = 1 AND locked_label = 0) AS changed_0_to_1,
  COUNTIF(alternative_label = 0 AND locked_label = 1) AS changed_1_to_0
FROM classified
GROUP BY method
ORDER BY method
'''

reference_sensitivity = client.query(
    sql_reference_sensitivity,
    location=BQ_LOCATION,
).to_dataframe()

display(reference_sensitivity)
save_csv(reference_sensitivity, "41A_reference_creatinine_label_sensitivity.csv")

## 03 — Landmark inclusion/exclusion characterization

Groups are assigned using a fixed precedence. Only aggregate summaries are saved.

In [ ]:
sql_landmark_groups = f'''
WITH selected AS (
  SELECT
    c.*,
    CASE
      WHEN c.eligible_main_cohort = 1 THEN 'included_main'
      WHEN c.reference_creatinine IS NULL THEN 'no_reference_creatinine'
      WHEN c.strict_chronic_dialysis_esrd = 1 THEN 'chronic_dialysis_or_esrd'
      WHEN COALESCE(c.max_stage_by_12h, 0) >= 2 THEN 'severe_aki_by_12h'
      WHEN c.first_acute_rrt_strict_offset IS NOT NULL
        AND c.first_acute_rrt_strict_offset <= 720 THEN 'acute_rrt_by_12h'
      WHEN c.observation_class = 'indeterminate_early_death' THEN 'early_death'
      WHEN c.observation_class = 'indeterminate_no_future_creatinine'
        THEN 'no_future_creatinine'
      WHEN c.observation_class = 'indeterminate_inadequate_followup'
        THEN 'inadequate_followup'
      ELSE 'other_not_in_main_cohort'
    END AS analysis_group
  FROM `{COHORT_TABLE}` c
  WHERE c.deterministic_eligible_patient_stay_rank = 1
),
joined AS (
  SELECT
    s.*,
    CASE WHEN p.age = '> 89' THEN 90 ELSE SAFE_CAST(p.age AS INT64) END AS age_years,
    LOWER(TRIM(p.gender)) = 'female' AS is_female,
    LOWER(TRIM(s.unit_discharge_status)) = 'expired' AS died_in_icu,
    SAFE_DIVIDE(s.unitDischargeOffset, 60.0) AS icu_length_hours
  FROM selected s
  LEFT JOIN `{PATIENT_TABLE}` p USING(patientUnitStayID)
)
SELECT
  analysis_group,
  COUNT(*) AS patients,
  COUNTIF(outcome_creatinine_stage23 = 1) AS observed_stage23_events,
  AVG(age_years) AS mean_age,
  APPROX_QUANTILES(age_years, 100)[OFFSET(50)] AS median_age,
  AVG(CAST(is_female AS INT64)) AS female_fraction,
  AVG(CAST(died_in_icu AS INT64)) AS icu_mortality_fraction,
  APPROX_QUANTILES(icu_length_hours, 100)[OFFSET(50)] AS median_icu_hours,
  AVG(reference_creatinine) AS mean_reference_creatinine,
  AVG(CAST(stage1_at_prediction AS INT64)) AS stage1_fraction
FROM joined
GROUP BY analysis_group
ORDER BY patients DESC
'''

landmark_groups = client.query(
    sql_landmark_groups,
    location=BQ_LOCATION,
).to_dataframe()

display(landmark_groups)
save_csv(landmark_groups, "41B_landmark_inclusion_exclusion_characteristics.csv")

## 04 — Prespecified operating points and alarm burden

No threshold is optimized on the outer-test predictions. The table uses prespecified probability thresholds aligned with the locked DCA range.

In [ ]:
sql_predictions = f'''
SELECT
  p.id_row,
  p.outer_fold,
  p.label_stage23,
  p.prediction_platt,
  c.group_hospital
FROM `{PRED_TABLE}` p
JOIN `{CORE_TABLE}` c USING(id_row)
'''
pred = client.query(sql_predictions, location=BQ_LOCATION).to_dataframe(
    create_bqstorage_client=True
)

assert len(pred) == EXPECTED_N
assert int(pred["label_stage23"].sum()) == EXPECTED_EVENTS
assert pred["group_hospital"].nunique() == EXPECTED_HOSPITALS

THRESHOLDS = [0.01, 0.02, 0.03, 0.05, 0.075, 0.10]

def operating_metrics(y, p, threshold):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pred_pos = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred_pos, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) else np.nan
    return {
        "threshold": threshold,
        "patients": len(y),
        "events": int(y.sum()),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "alerts_per_100_patients": 100 * (tp + fp) / len(y),
        "true_alerts_per_100_patients": 100 * tp / len(y),
        "false_alerts_per_100_patients": 100 * fp / len(y),
        "patients_evaluated_per_true_positive": (tp + fp) / tp if tp else np.nan,
    }

operating = pd.DataFrame([
    operating_metrics(
        pred["label_stage23"],
        pred["prediction_platt"],
        t
    )
    for t in THRESHOLDS
])
display(operating)
save_csv(operating, "41C_prespecified_operating_points.csv")

# Optional hospital-cluster bootstrap confidence intervals.
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 20260805
rng = np.random.default_rng(BOOTSTRAP_SEED)

hospital_groups = {
    h: g[["label_stage23", "prediction_platt"]].reset_index(drop=True)
    for h, g in pred.groupby("group_hospital", sort=False)
}
hospitals = np.array(list(hospital_groups.keys()), dtype=object)

boot_rows = []
for t in THRESHOLDS:
    reps = []
    for _ in range(BOOTSTRAP_REPS):
        sampled = rng.choice(hospitals, size=len(hospitals), replace=True)
        blocks = [hospital_groups[h] for h in sampled]
        b = pd.concat(blocks, ignore_index=True)
        m = operating_metrics(b["label_stage23"], b["prediction_platt"], t)
        reps.append(m)
    rep_df = pd.DataFrame(reps)
    for metric in [
        "sensitivity", "specificity", "ppv", "npv", "f1",
        "alerts_per_100_patients", "false_alerts_per_100_patients"
    ]:
        boot_rows.append({
            "threshold": t,
            "metric": metric,
            "bootstrap_replicates": BOOTSTRAP_REPS,
            "point_estimate": float(operating.loc[operating["threshold"] == t, metric].iloc[0]),
            "ci_95_lower": float(rep_df[metric].quantile(0.025)),
            "ci_95_upper": float(rep_df[metric].quantile(0.975)),
            "bootstrap_unit": "hospital",
            "seed": BOOTSTRAP_SEED,
        })

operating_ci = pd.DataFrame(boot_rows)
display(operating_ci.head(20))
save_csv(operating_ci, "41D_operating_points_hospital_bootstrap_95CI.csv")

## 05 — Hospital-level performance and missingness heterogeneity

AUROC/AUPRC are calculated only for hospitals with at least 20 events and at least 20 non-events. All hospitals receive event-rate, mean-risk and calibration-gap summaries.

In [ ]:
hospital_summary = (
    pred.groupby("group_hospital")
    .agg(
        patients=("label_stage23", "size"),
        events=("label_stage23", "sum"),
        event_rate=("label_stage23", "mean"),
        mean_predicted_risk=("prediction_platt", "mean"),
    )
    .reset_index()
)
hospital_summary["nonevents"] = hospital_summary["patients"] - hospital_summary["events"]
hospital_summary["calibration_gap"] = (
    hospital_summary["mean_predicted_risk"] - hospital_summary["event_rate"]
)

perf_rows = []
for hospital, g in pred.groupby("group_hospital"):
    events = int(g["label_stage23"].sum())
    nonevents = int(len(g) - events)
    if events >= 20 and nonevents >= 20:
        perf_rows.append({
            "group_hospital": hospital,
            "patients": len(g),
            "events": events,
            "event_rate": g["label_stage23"].mean(),
            "mean_predicted_risk": g["prediction_platt"].mean(),
            "calibration_gap": g["prediction_platt"].mean() - g["label_stage23"].mean(),
            "auroc": roc_auc_score(g["label_stage23"], g["prediction_platt"]),
            "auprc": average_precision_score(g["label_stage23"], g["prediction_platt"]),
        })

hospital_perf = pd.DataFrame(perf_rows)
display(hospital_summary.describe(include="all"))
display(hospital_perf.describe())

save_csv(hospital_summary, "41E_all_hospital_risk_and_calibration_summary.csv")
save_csv(hospital_perf, "41F_hospitals_with_20plus_events_performance.csv")

# Load only the locked matrix required for missingness heterogeneity.
core = client.query(
    f"SELECT * FROM `{CORE_TABLE}`",
    location=BQ_LOCATION,
).to_dataframe(create_bqstorage_client=True)

predictor_cols = [c for c in core.columns if c.startswith("x_")]
assert len(predictor_cols) == 159

missing_rows = []
for feature in predictor_cols:
    by_hospital = core.groupby("group_hospital")[feature].apply(lambda s: s.isna().mean())
    missing_rows.append({
        "feature": feature,
        "overall_missing_rate": core[feature].isna().mean(),
        "hospital_missing_rate_q25": by_hospital.quantile(0.25),
        "hospital_missing_rate_median": by_hospital.quantile(0.50),
        "hospital_missing_rate_q75": by_hospital.quantile(0.75),
        "hospital_missing_rate_min": by_hospital.min(),
        "hospital_missing_rate_max": by_hospital.max(),
    })

missing_heterogeneity = pd.DataFrame(missing_rows).sort_values(
    ["hospital_missing_rate_q75", "overall_missing_rate"],
    ascending=False,
)
display(missing_heterogeneity.head(30))
save_csv(missing_heterogeneity, "41G_predictor_missingness_heterogeneity_by_hospital.csv")

## 06 — Aggregate figures

In [ ]:
# Hospital-disjoint fold performance
fold_summary = (
    pred.groupby("outer_fold")
    .apply(lambda g: pd.Series({
        "patients": len(g),
        "events": int(g["label_stage23"].sum()),
        "event_rate": g["label_stage23"].mean(),
        "auroc": roc_auc_score(g["label_stage23"], g["prediction_platt"]),
        "auprc": average_precision_score(g["label_stage23"], g["prediction_platt"]),
    }))
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.plot(fold_summary["outer_fold"], fold_summary["auroc"], marker="o", label="AUROC")
plt.plot(fold_summary["outer_fold"], fold_summary["auprc"], marker="o", label="AUPRC")
plt.xticks(fold_summary["outer_fold"])
plt.xlabel("Hospital-disjoint outer fold")
plt.ylabel("Performance")
plt.title("Outer-fold performance")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "41H_outer_fold_performance.png", dpi=220)
plt.show()

# Hospital event rate versus mean predicted risk
plt.figure(figsize=(6.5, 6.5))
plt.scatter(
    hospital_summary["event_rate"],
    hospital_summary["mean_predicted_risk"],
    s=np.maximum(10, np.sqrt(hospital_summary["patients"]) * 2),
    alpha=0.65,
)
upper = max(
    hospital_summary["event_rate"].max(),
    hospital_summary["mean_predicted_risk"].max(),
)
plt.plot([0, upper], [0, upper], linestyle="--")
plt.xlabel("Observed hospital event rate")
plt.ylabel("Mean predicted hospital risk")
plt.title("Hospital-level calibration-in-the-large")
plt.tight_layout()
plt.savefig(OUT / "41I_hospital_event_rate_vs_mean_risk.png", dpi=220)
plt.show()

# Operating-point alarm burden
plt.figure(figsize=(8, 5))
plt.plot(
    operating["threshold"],
    operating["alerts_per_100_patients"],
    marker="o",
    label="All alerts"
)
plt.plot(
    operating["threshold"],
    operating["false_alerts_per_100_patients"],
    marker="o",
    label="False alerts"
)
plt.xlabel("Probability threshold")
plt.ylabel("Alerts per 100 patients")
plt.title("Prespecified operating-point alarm burden")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "41J_operating_point_alarm_burden.png", dpi=220)
plt.show()

save_csv(fold_summary, "41K_reconstructed_fold_summary.csv")

## 07 — Manifest and final checks

In [ ]:
expected_outputs = [
    "41A_reference_creatinine_label_sensitivity.csv",
    "41B_landmark_inclusion_exclusion_characteristics.csv",
    "41C_prespecified_operating_points.csv",
    "41D_operating_points_hospital_bootstrap_95CI.csv",
    "41E_all_hospital_risk_and_calibration_summary.csv",
    "41F_hospitals_with_20plus_events_performance.csv",
    "41G_predictor_missingness_heterogeneity_by_hospital.csv",
    "41H_outer_fold_performance.png",
    "41I_hospital_event_rate_vs_mean_risk.png",
    "41J_operating_point_alarm_burden.png",
    "41K_reconstructed_fold_summary.csv",
]

manifest_rows = []
for name in expected_outputs:
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(path)
    manifest_rows.append({
        "filename": name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

manifest = pd.DataFrame(manifest_rows)
display(manifest)
save_csv(manifest, "41L_strengthening_manifest.csv")

manifest_json = {
    "analysis": "AKI_V2_targeted_strengthening",
    "version": "v001",
    "primary_model_refit": False,
    "retuning": False,
    "patient_level_data_written_to_drive": False,
    "aggregate_outputs": expected_outputs,
    "expected_primary_patients": EXPECTED_N,
    "expected_primary_events": EXPECTED_EVENTS,
    "expected_hospitals": EXPECTED_HOSPITALS,
}
with open(OUT / "41L_strengthening_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest_json, f, indent=2)

print("TARGETED STRENGTHENING RUN: COMPLETE")
print("Outputs:", OUT)